In [34]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
%matplotlib inline

In [35]:
# Load data
shot_df = pd.read_csv('shot_data.csv')

In [36]:
# Prepare data
for col in ['rebound', 'overtime', 'shooting_on_empty', 'goal', 'season']:
    shot_df[col] = shot_df[col].astype(int)

X = shot_df.drop(['goal'], axis=1)
y = shot_df['goal'].values

In [37]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=87,
    stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 363513 | Test: 90879


In [38]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['shot_type'])
    ],
    remainder='passthrough'  # keeps numeric columns unchanged
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [39]:
# Split test data
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train_encoded, y_train,
    test_size=0.2,
    random_state=87,
    stratify=y_train
)

In [40]:
# scale
neg = (y_train_part == 0).sum()
pos = (y_train_part == 1).sum()
scale = neg / pos

In [41]:
# Optimize Parameters - 1
'''
param_distributions = {
    'learning_rate': [0.1, 0.01, 0.001],
    'max_depth': [4, 6, 8],
    'gamma': [0, 0.25, 1.0],
    'reg_lambda': [0.1, 1.0, 10.0],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7]
}

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective='binary:logistic',
        n_estimators=200, # set n_estimators for parameter optimization. set higher in final model with early_stopping
        random_state=87,
        scale_pos_weight=scale,
        eval_metric='aucpr'
    ),
    param_distributions=param_distributions,
    n_iter=50, # number of random combinations to try
    scoring='average_precision',
    cv=4,
    verbose=2,
    random_state=87,
    n_jobs=-1
)

random_search.fit(X_train_encoded, y_train)

print(random_search.best_params_)
'''
# {'subsample': 0.9, 'reg_lambda': 10.0, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.25, 'colsample_bytree': 0.5}

"\nparam_distributions = {\n    'learning_rate': [0.1, 0.01, 0.001],\n    'max_depth': [4, 6, 8],\n    'gamma': [0, 0.25, 1.0],\n    'reg_lambda': [0.1, 1.0, 10.0],\n    'subsample': [0.7, 0.8, 0.9],\n    'colsample_bytree': [0.5, 0.6, 0.7]\n}\n\nrandom_search = RandomizedSearchCV(\n    estimator=XGBClassifier(\n        objective='binary:logistic',\n        n_estimators=200, # set n_estimators for parameter optimization. set higher in final model with early_stopping\n        random_state=87,\n        scale_pos_weight=scale,\n        eval_metric='aucpr'\n    ),\n    param_distributions=param_distributions,\n    n_iter=50, # number of random combinations to try\n    scoring='average_precision',\n    cv=4,\n    verbose=2,\n    random_state=87,\n    n_jobs=-1\n)\n\nrandom_search.fit(X_train_encoded, y_train)\n\nprint(random_search.best_params_)\n"

In [42]:
# Optimize Parameters - 2
'''
param_distributions = {
    'learning_rate': [0.1, 0.2, 0.3],
    'max_depth': [2, 3, 4],
    'gamma': [0.1, 0.25, 0.4],
    'reg_lambda': [10.0, 15.0, 20.0],
    'subsample': [0.9, 0.95, 1.0],
    'colsample_bytree': [0.3, 0.4, 0.5]
}

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective='binary:logistic',
        n_estimators=200, # set n_estimators for parameter optimization. set higher in final model with early_stopping
        random_state=87,
        scale_pos_weight=scale,
        eval_metric='aucpr'
    ),
    param_distributions=param_distributions,
    n_iter=50, # number of random combinations to try
    scoring='average_precision',
    cv=4,
    verbose=2,
    random_state=87,
    n_jobs=-1
)

random_search.fit(X_train_encoded, y_train)

print(random_search.best_params_)
'''
# {'subsample': 0.95, 'reg_lambda': 15.0, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.25, 'colsample_bytree': 0.4}

"\nparam_distributions = {\n    'learning_rate': [0.1, 0.2, 0.3],\n    'max_depth': [2, 3, 4],\n    'gamma': [0.1, 0.25, 0.4],\n    'reg_lambda': [10.0, 15.0, 20.0],\n    'subsample': [0.9, 0.95, 1.0],\n    'colsample_bytree': [0.3, 0.4, 0.5]\n}\n\nrandom_search = RandomizedSearchCV(\n    estimator=XGBClassifier(\n        objective='binary:logistic',\n        n_estimators=200, # set n_estimators for parameter optimization. set higher in final model with early_stopping\n        random_state=87,\n        scale_pos_weight=scale,\n        eval_metric='aucpr'\n    ),\n    param_distributions=param_distributions,\n    n_iter=50, # number of random combinations to try\n    scoring='average_precision',\n    cv=4,\n    verbose=2,\n    random_state=87,\n    n_jobs=-1\n)\n\nrandom_search.fit(X_train_encoded, y_train)\n\nprint(random_search.best_params_)\n"

In [43]:
best_params = {
    'learning_rate': 0.1,
    'max_depth': 4,
    'gamma': 0.25,
    'reg_lambda': 15.0,
    'subsample': 0.95,
    'colsample_bytree': 0.4
}

model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=1000,
    scale_pos_weight=scale,
    random_state=87,
    eval_metric='aucpr',
    early_stopping_rounds=50,
    **best_params
)

model.fit(X_train_part,
          y_train_part,
          verbose=True,
          eval_set=[(X_val, y_val)])

[0]	validation_0-aucpr:0.13587
[1]	validation_0-aucpr:0.18456
[2]	validation_0-aucpr:0.19100
[3]	validation_0-aucpr:0.18992
[4]	validation_0-aucpr:0.20027
[5]	validation_0-aucpr:0.20090
[6]	validation_0-aucpr:0.20787
[7]	validation_0-aucpr:0.21036
[8]	validation_0-aucpr:0.21238
[9]	validation_0-aucpr:0.21370
[10]	validation_0-aucpr:0.21327
[11]	validation_0-aucpr:0.21462
[12]	validation_0-aucpr:0.21714
[13]	validation_0-aucpr:0.21668
[14]	validation_0-aucpr:0.21634
[15]	validation_0-aucpr:0.21722
[16]	validation_0-aucpr:0.21991
[17]	validation_0-aucpr:0.22103
[18]	validation_0-aucpr:0.21988
[19]	validation_0-aucpr:0.22055
[20]	validation_0-aucpr:0.22019
[21]	validation_0-aucpr:0.21868
[22]	validation_0-aucpr:0.21827
[23]	validation_0-aucpr:0.21955
[24]	validation_0-aucpr:0.21994
[25]	validation_0-aucpr:0.22097
[26]	validation_0-aucpr:0.22070
[27]	validation_0-aucpr:0.22096
[28]	validation_0-aucpr:0.22099
[29]	validation_0-aucpr:0.22149
[30]	validation_0-aucpr:0.22158
[31]	validation_0-

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.4
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=

In [44]:
# AUC
probs = model.predict_proba(X_test_encoded)[:, 1]
auc = roc_auc_score(y_test, probs)
print(f"AUC: {auc:.3f}")

AUC: 0.782


In [45]:
# AUCPR
aucpr = average_precision_score(y_test, probs)
print(f"AUCPR: {aucpr:.3f}")

AUCPR: 0.245


In [46]:
# Accuracy
preds = model.predict(X_test_encoded)
correct = (preds == y_test).sum()
print(f"Accuracy: {correct}/{len(y_test)} ({correct/len(y_test)*100:.1f}%)")

Accuracy: 60489/90879 (66.6%)


## Final Model

In [47]:
X_full_encoded = preprocessor.transform(X)

neg = (y == 0).sum()
pos = (y == 1).sum()
scale = neg / pos

final_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=model.best_iteration,
    scale_pos_weight=scale,
    random_state=87,
    eval_metric='aucpr',
    **best_params
)

final_model.fit(X_full_encoded, y)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.4
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [ ]:
# calibrate so probabilities center around average goals/shot
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(final_model, method='isotonic')
calibrated_model.fit(X_full_encoded, y)

probs_calibrated = calibrated_model.predict_proba(X_test_encoded)[:, 1]
print(probs_calibrated.mean())  # should be close to 0.07

0.07188526194780304


In [ ]:
# save model
# import os
# import joblib

# os.makedirs("models", exist_ok=True)

# joblib.dump(calibrated_model, "streamlit-app/models/xgb_model.pkl")
# joblib.dump(preprocessor, "streamlit-app/models/preprocessor.pkl") # also save preprocessor. encoding must be consistent for model to work

['streamlit-app/models/preprocessor.pkl']